<a href="https://colab.research.google.com/github/satyam-1605/RAG-from-basic-to-advance/blob/main/LCEL(Langchain_Expression_Language).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain_google_genai
!pip install langchain_community
!pip install langchain
!pip install langchain_huggingface
!pip install langchain_groq

In [ ]:
from google.colab import userdata
GROQ_API_KEY=userdata.get('GROQ_API_KEY')
import os
os.environ["GROQ_API_KEY"]=GROQ_API_KEY

In [ ]:
from google.colab import userdata
GEMINI_API_KEY=userdata.get('GEMINI_API_KEY')
import os
os.environ["GEMINI_API_KEY"]=GEMINI_API_KEY

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [ ]:
'''from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
from langchain_groq import ChatGroq
import os
llm=ChatGroq(model_name="Gemma2-9b-It")'''

# this is my simple chain (old chaining concept)

In [ ]:
template= 'Hi! I am learning {skill}. Can you suggest me top 5 things to learn?\n'

In [ ]:
from langchain_core.prompts import PromptTemplate

In [ ]:
prompt = PromptTemplate(template=template,input_variables=["skill"])

In [ ]:
print(prompt)

In [ ]:
from langchain_classic import LLMChain

In [ ]:
llm_chain = LLMChain(prompt=prompt,llm=llm)

In [ ]:
print(llm_chain.run('Data Science'))

# this is a implementation  using LCEL

In [ ]:
llm

In [ ]:
prompt

In [ ]:
chain = prompt | llm

In [ ]:
print(chain.invoke({'skill':'Big Data'}))

In [ ]:
from langchain_core.output_parsers import StrOutputParser

In [ ]:
parser = StrOutputParser()

In [ ]:
chain = prompt | llm | parser

In [ ]:
print(chain.invoke({'skill':'Machine Learning'}))

# lets discuss about the runnables

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough , RunnableLambda

In [ ]:
chain = RunnablePassthrough()

In [ ]:
chain.invoke('Welcome to this youtube channel')

In [ ]:
chain = RunnablePassthrough() | RunnablePassthrough() | RunnablePassthrough()

In [ ]:
chain.invoke('Welcome to my sunny"s youtube channel')

In [ ]:
def string_upper(input):
  return input.upper()

In [ ]:
chain = RunnablePassthrough() | RunnableLambda(string_upper)

In [ ]:
chain.invoke('Welcome to my sunny"s youtube channel')

In [ ]:
chain = RunnableLambda(string_upper)

In [ ]:
chain.invoke('Welcome to my sunny"s youtube channel')

In [ ]:
chain = RunnableParallel({'x':RunnablePassthrough(),'y':RunnablePassthrough()})

In [ ]:
chain.invoke("Sunny")

In [ ]:
chain.invoke({'Youtube': '@sunnysavita10','Blog': "Sunny's blog"})

In [ ]:
lambda x: x['Blog']

In [ ]:
chain = RunnableParallel({'x':RunnablePassthrough(),'Blog':lambda x: x['Blog']})

In [ ]:
chain.invoke({'Youtube': '@sunnysavita10','Blog': "Sunny's blog"})

In [ ]:
def fetch_website(input: dict):
    output = input.get('Website','Not found')
    return output

In [ ]:
mydict={'Youtube': '@sunnysavita10','Blog': "Sunny's blog"}

In [ ]:
mydict.get("website","Not found")

In [ ]:
chain = RunnableParallel({'Website':RunnablePassthrough() | RunnableLambda(fetch_website),
                          'Blog':lambda z: z['Blog']})

In [ ]:
chain.invoke({'Youtube': '@sunnysavita10','Blog': "Sunny's blog"})

In [ ]:
chain.invoke({'Youtube': '@sunnysavita10','Blog': "Sunny's blog" , 'Website' : 'sunnysavita.com'})

In [ ]:
def extra_func(input):
    return 'Happy Learning'

In [ ]:
chain = RunnableParallel({'x' : RunnablePassthrough()}).assign(extra=RunnableLambda(extra_func))

In [ ]:
chain = RunnableParallel({'x' : RunnablePassthrough()}).assign(y=RunnableLambda(extra_func))

In [ ]:
chain.invoke('Hello')

In [ ]:
!pip install --upgrade chromadb opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp-proto-grpc

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

### Reading the txt files from source directory

loader = DirectoryLoader('/content/', glob="*.txt", loader_cls=TextLoader)
docs = loader.load()

### Creating Chunks using RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10,
    length_function=len
)
new_docs = text_splitter.split_documents(documents=docs)
doc_strings = [doc.page_content for doc in new_docs]

###  BGE Embddings

'''from langchain.embeddings import HuggingFaceBgeEmbeddings

model_name = "BAAI/bge-base-en-v1.5"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': True} # set True to compute cosine similarity
embeddings = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)
'''

### Creating Retriever using Vector DB

db = Chroma.from_documents(new_docs, embeddings)
retriever = db.as_retriever(search_kwargs={"k": 4})

In [ ]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = PromptTemplate.from_template(template)


In [ ]:
retrieval_chain = (
    RunnableParallel({"context": retriever, "question": RunnablePassthrough()})
    | prompt
    | llm
    | StrOutputParser()
    )

In [ ]:
question ="Why did Biden claim Putin “badly miscalculated”?"

In [ ]:
retrieval_chain.invoke(question)

In [ ]:
import time

start_time = time.time()

result = retrieval_chain.invoke(question)

print('Time taken:',time.time() - start_time)

In [ ]:
start_time = time.time()

result = retrieval_chain.ainvoke(question)

print('Time taken:',time.time() - start_time)

In [ ]:
start_time = time.time()

batch_output = retrieval_chain.batch([
                        "How did the Ukraine conflict affect Biden’s discussion of the U.S. economy",
                        "can you highlight 3 main properties?"
                       ])

print('Time taken:',time.time() - start_time)

In [ ]:
batch_output

In [ ]:
start_time = time.time()

batch_output = await retrieval_chain.abatch([
                        "what is llama3?",
                        "can you highlight 3 main properties?"
                       ])

print('Time taken:',time.time() - start_time)

In [ ]:
batch_output

In [ ]:
my_dict = {'Youtube': '@sunnysavita10','Blog': "sunny's blog" , 'Website' : 'sunnysavita.com'}
my_dict

In [ ]:
from operator import itemgetter

website = itemgetter('Website')

In [ ]:
website(my_dict)

In [ ]:
template = """Answer the question based only on the following context:
{context}

Question: {question}

Answer in the following language: {language}
"""
prompt = PromptTemplate.from_template(template)


In [ ]:
retrieval_chain = (
    RunnableParallel({"context": itemgetter('question') | retriever,
                       "question": itemgetter('question'),
                       "language": itemgetter('language')
                       })
    | prompt
    | llm
    | StrOutputParser()
    )

In [ ]:
### itemgetter only works with dictionaries , input has to be a dict

response = retrieval_chain.invoke({'question': "what is llama3?",
                        'language': "Spnish"})

print(response)

In [ ]:
template = 'Hi! I am learning {skill}. Can you suggest me top 5 things to learn?\n'

prompt = PromptTemplate.from_template(template=template)

chain = prompt | llm

In [ ]:
for s in chain.stream({'skill':'Big Data'}):
    print(s.content,end='')